In [2]:
!pip install transformers datasets scikit-learn joblib tqdm matplotlib seaborn

import os
import re
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5EncoderModel
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


if not os.path.exists('t5_checkpoints/plots'):
    os.makedirs('t5_checkpoints/plots')

print(f"Zariadenie: {device}")

Zariadenie: cuda


In [13]:
df = pd.read_csv('labeled_data.csv')

def clean_text(text):
    
    text = text.lower()
    text = re.sub(r'@[\w\-]+', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\brt\b', '', text)
    text = re.sub(r'&amp;|&#\d+;', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['clean_tweet'] = df['tweet'].apply(clean_text)

df = df.drop_duplicates(subset=['clean_tweet'])

df = df[df['clean_tweet'] != ''].dropna()

X = df['clean_tweet']
y = df['class']

print(f"Celkový počet záznamov po čistení: {len(df)}")

Celkový počet záznamov po čistení: 24443


In [14]:
class SimpleDataset(Dataset):
    
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts.values
        self.labels = labels.values
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, 
            padding='max_length', 
            truncation=True, 
            max_length=64, 
            return_tensors='pt'
        )
        
        return{
            'ids': encoding['input_ids'].flatten(),
            'mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [15]:
class T5Classifier(nn.Module):
    def __init__(self, num_classes=3):
        super(T5Classifier, self).__init__()
        self.encoder = T5EncoderModel.from_pretrained("t5-small") 
        self.classifier = nn.Linear(self.encoder.config.d_model, num_classes)

    def forward(self, ids, mask):
        outputs = self.encoder(input_ids=ids, attention_mask=mask)
        mean_output = torch.mean(outputs.last_hidden_state, dim=1)
        logits = self.classifier(mean_output)
        return logits

In [16]:
def save_fold_loss_curve(history, fold_num):
   
    plt.figure(figsize=(4, 2.5))
    
    plt.plot(history['train'], label='Train Loss', linewidth=1.5, marker='o', markersize=4)
    
    plt.plot(history['val'], label='Val Loss', linewidth=1.5, marker='s', markersize=4)
    
    plt.title(f'Loss: Fold {fold_num}', fontsize=9)
    plt.xlabel('Epocha', fontsize=8)
    plt.ylabel('Loss', fontsize=8)
    plt.xticks(range(len(history['train'])), range(1, len(history['train']) + 1), fontsize=7)
    plt.yticks(fontsize=7)
    plt.legend(fontsize=7)
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plot_path = f't5_checkpoints/plots/fold_{fold_num}_loss_curves.png'
    
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Graf uložený: {plot_path}")

In [17]:
def train_model_advanced (model, train_loader, val_loader, fold, epochs=5, patience=2):
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
    criterion = torch.nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    epochs_no_improve = 0  
    fold_history = {'train': [], 'val': []}

    print(f"Trénovanie Fold {fold} na zariadení {device}")
    
    for epoch in range(epochs):
        # trenovanie
        model.train()
        total_train_loss = 0
        train_loop = tqdm(train_loader, desc=f"Fold {fold} | Ep {epoch+1}/{epochs}")
        
        for batch in train_loop:
            optimizer.zero_grad()
            ids = batch['ids'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(ids, mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
            train_loop.set_postfix(loss=loss.item())

        #validacia
        
        model.eval()
        total_val_loss = 0
        correct = 0
        
        with torch.no_grad():
            
            for batch in val_loader:
                
                ids = batch['ids'].to(device)
                mask = batch['mask'].to(device)
                labels = batch['label'].to(device)
                
                outputs = model(ids, mask)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels).sum().item()

        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)
        accuracy = correct / len(val_loader.dataset)
        
        fold_history['train'].append(avg_train_loss)
        fold_history['val'].append(avg_val_loss)
        
        loss_diff = abs(avg_val_loss - avg_train_loss)
        percent_diff = (loss_diff / avg_train_loss * 100) if avg_train_loss != 0 else 0

        print(f"\n[EPOCHA {epoch+1}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Accuracy: {accuracy:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f"t5_checkpoints/best_t5_model_fold_{fold}.pt")
            epochs_no_improve = 0
            
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early Stopping v epoche {epoch+1}")
            break
        print("-" * 20)
        
    return fold_history

In [18]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tokenizer = T5Tokenizer.from_pretrained("t5-small")

global_best_loss = float('inf')
najlepsi_fold_cislo = 1
best_val_indices = None  

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n==================== FOLD {fold+1} ====================")
    
    train_loader = DataLoader(SimpleDataset(X.iloc[train_idx], y.iloc[train_idx], tokenizer), batch_size=8, shuffle=True)
    val_loader = DataLoader(SimpleDataset(X.iloc[val_idx], y.iloc[val_idx], tokenizer), batch_size=8)
    
    fold_model = T5Classifier().to(device)
    
    current_history = train_model_advanced(fold_model, train_loader, val_loader, fold=fold+1, epochs=5, patience=2)
    save_fold_loss_curve(current_history, fold_num=fold+1)
    
    path_to_current_best = f"t5_checkpoints/best_t5_model_fold_{fold+1}.pt"
    fold_model.load_state_dict(torch.load(path_to_current_best))
    
    fold_model.eval()
    current_fold_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            logits = fold_model(batch['ids'].to(device), batch['mask'].to(device))
            loss = torch.nn.CrossEntropyLoss()(logits, batch['label'].to(device))
            current_fold_loss += loss.item()
    
    avg_current_fold_loss = current_fold_loss / len(val_loader)
    print(f"Finálna validačná strata pre Fold {fold+1}: {avg_current_fold_loss:.4f}")
    
    if avg_current_fold_loss < global_best_loss:
        global_best_loss = avg_current_fold_loss
        najlepsi_fold_cislo = fold + 1
        best_val_indices = val_idx
        torch.save(fold_model.state_dict(), "t5_checkpoints/absolute_best_t5_model.pt")
        print(f"--> Nový najlepší model nájdený vo Folde {fold+1}!")


==================== FOLD 1 ====================


Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

Trénovanie Fold 1 na zariadení cuda


Fold 1 | Ep 1/5:   0%|          | 0/2445 [00:00<?, ?it/s]


[EPOCHA 1] Train Loss: 0.4522 | Val Loss: 0.2992 | Accuracy: 0.9057
--------------------


...


[EPOCHA 3] Train Loss: 0.2622 | Val Loss: 0.2857 | Accuracy: 0.9055
--------------------


Fold 5 | Ep 4/5:   0%|          | 0/2445 [00:00<?, ?it/s]


[EPOCHA 4] Train Loss: 0.2397 | Val Loss: 0.2857 | Accuracy: 0.9041
--------------------


Fold 5 | Ep 5/5:   0%|          | 0/2445 [00:00<?, ?it/s]


[EPOCHA 5] Train Loss: 0.2236 | Val Loss: 0.2820 | Accuracy: 0.9059
--------------------
Graf uložený: t5_checkpoints/plots/fold_5_loss_curves.png
Finálna validačná strata pre Fold 5: 0.2820


In [27]:
def get_detailed_report(model, loader, title="VÝSLEDOK"):
    
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            ids = batch['ids'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(ids, mask)
            preds = torch.argmax(logits, dim=1)
               
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    print(f"\n=== {title} ===")
    print(classification_report(all_labels, all_preds, 
                                target_names=['Hate Speech', 'Offensive', 'Neither'], 
                                zero_division=0,digits=4))
    return all_labels, all_preds

In [28]:
final_best_model = T5Classifier().to(device)
model_path = "t5_checkpoints/absolute_best_t5_model.pt"

try:
    final_best_model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Najlepší model z Foldu {najlepsi_fold_cislo} bol úspešne načítaný.")
    
    final_val_loader = DataLoader(SimpleDataset(X.iloc[best_val_indices], y.iloc[best_val_indices], tokenizer), batch_size=8)
    
    # Vygenerovanie reportu
    y_true_nn, y_pred_nn = get_detailed_report(
        final_best_model, 
        final_val_loader, 
        title=f"FINÁLNY REPORT: T5-Small (Fold {najlepsi_fold_cislo})"
    )
    
    
except FileNotFoundError:
    print(f"Chyba: Súbor {model_path} nebol nájdený.")

Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

Najlepší model z Foldu 1 bol úspešne načítaný.

=== FINÁLNY REPORT: T5-Small (Fold 1) ===
              precision    recall  f1-score   support

 Hate Speech     0.5526    0.3723    0.4449       282
   Offensive     0.9453    0.9593    0.9523      3786
     Neither     0.8786    0.9172    0.8975       821

    accuracy                         0.9184      4889
   macro avg     0.7922    0.7496    0.7649      4889
weighted avg     0.9115    0.9184    0.9138      4889

